### Event Data vs Snapshot Data
#### Event data (many rows per entity)
```text
user_id | event_time | amount
```
Examples:
- orders
- transactions
- clicks
- logs

#### Snapshot data (one row per entity per time)
```text
user_id | snapshot_time | features
```
ML **always trains on snapshots**, not raw events.

### The Time Cutoff
A cutoff defines the boundary between:
- past (allowed)
- future (forbidden)
```python
cutoff_time = pd.Timestamp("2023-02-01")
```

### Common Time Leakage Patterns
#### Leakage 1 — Aggregating full history
```python
orders.groupby("user_id")["amount"].sum()
```

Uses future orders → invalid model.

#### Leakage 2 — Random train/test split
```python
train_test_split(df)
```

Future data leaks into training.

#### Leakage 3 — Rolling without cutoff
```python
rolling("30D")
```

Without cutoff, rolling windows see the future.

#### Leakage 4 — Label leakage
```python
df["churn_next_30d"]
```

Used directly in feature logic.

### Correct Time-Aware Train/Test Split
#### Wrong
```python
train_test_split(df, test_size=0.2)
```

#### Correct
```python
train = df[df["snapshot_time"] < "2023-03-01"]
test  = df[df["snapshot_time"] >= "2023-03-01"]
```

### Rolling vs Expanding Windows
#### Rolling window (short-term behavior)
```python
rolling("30D")
```
- Recent activity
- Fast behavior change
- High leakage risk

#### Expanding window (long-term behavior)
```python
expanding()
```
- Lifetime behavior
- Stable patterns
- Lower leakage risk

**Example**
```python
orders["expanding_spend"] = (
    orders
    .groupby("user_id")["amount"]
    .expanding()
    .sum()
    .reset_index(level=0, drop=True)
)
```

### Lag Features
```python
orders["amount_lag_1"] = (
    orders
    .groupby("user_id")["amount"]
    .shift(1)
)
```
NaNs MUST appear

### Label Construction 
#### Example: churn in next 30 days
```python
label = (
    future_events
    .groupby("user_id")["event_time"]
    .min()
    < snapshot_time + pd.Timedelta(days=30)
)
```
Labels are built from the future, features from the past.

### Multiple Snapshots per Entity
Instead of one row per user:
```text
user_id | snapshot_time | features | label
```

This allows:
- online learning
- drift detection
- time-aware evaluation

Same user appears multiple times — this is correct.

### Feature Drift & Data Drift
Over time:
- feature distributions change
- model confidence degrades

Monitor:
- mean / std shifts
- missing rates
- category changes

### Base Data
#### User Signup Table

In [1]:
import pandas as pd
import numpy as np

users = pd.DataFrame({
    "user_id": [1, 2, 3],
    "signup_date": pd.to_datetime([
        "2023-01-01",
        "2023-01-05",
        "2023-01-10"
    ])
})

#### User Activity Events

In [2]:
events = pd.DataFrame({
    "event_id": range(101, 113),
    "user_id": [1,1,1, 2,2,2, 3,3,3,3,3,3],
    "event_time": pd.to_datetime([
        "2023-01-03", "2023-01-10", "2023-01-20",
        "2023-01-07", "2023-01-15", "2023-01-25",
        "2023-01-12", "2023-01-14", "2023-01-18",
        "2023-01-22", "2023-01-28", "2023-02-05"
    ]),
    "amount": [100, 200, 150, 50, 80, 120, 300, 400, 200, 100, 150, 180]
})

#### Snapshot Times

In [3]:
SNAPSHOT_DATES = [
    pd.Timestamp("2023-01-15"),
    pd.Timestamp("2023-01-22"),
    pd.Timestamp("2023-01-29")
]

### Exercise 1
Create a DataFrame with:
```text
user_id | snapshot_time
```

For **every user × every snapshot_time.**

In [5]:
snapshots = (
    users[["user_id"]]
    .assign(key=1)
    .merge(
        pd.DataFrame({"snapshot_time": SNAPSHOT_DATES, "key": 1}),
        on="key"
    )
    .drop(columns="key")
)
snapshots

,user_id,snapshot_time
0,1,2023-01-15
1,1,2023-01-22
2,1,2023-01-29
3,2,2023-01-15
4,2,2023-01-22
5,2,2023-01-29
6,3,2023-01-15
7,3,2023-01-22
8,3,2023-01-29


### Exercise 2
For each snapshot_time:
- Include only events where:
```text
event_time < snapshot_time
```
Explain why `<= snapshot_time` is unsafe.

The **events are filtered dynamically per snapshot**, but conceptually:
```text
event_time < snapshot_time
```
**Why NOT `<= snapshot_time`?**
- Events at the exact timestamp may arrive late
- Safer to treat snapshot_time as **prediction moment**
- `< snapshot_time` guarantees no future knowledge

### Exercise 3
For each snapshot:
- `total_amount`
- `event_count`
- `avg_amount`

Rules:
- Use **only filtered events**
- Users with no events -> NaN

In [6]:
feature_rows = []

for snapshot_time in SNAPSHOT_DATES:
    past_events = events[events["event_time"] < snapshot_time]

    agg = (
        past_events
        .groupby("user_id")
        .agg(
            total_amount = ("amount", "sum"),
            event_count = ("event_id", "count"),
            avg_amount = ("amount", "mean")
        )
        .reset_index()
    )

    agg["snapshot_time"] = snapshot_time
    feature_rows.append(agg)

agg_features = pd.concat(feature_rows, ignore_index=True)
agg_features

,user_id,total_amount,event_count,avg_amount,snapshot_time
0,1,300,2,150.000000,2023-01-15
1,2,50,1,50.000000,2023-01-15
2,3,700,2,350.000000,2023-01-15
3,1,450,3,150.000000,2023-01-22
4,2,130,2,65.000000,2023-01-22
5,3,900,3,300.000000,2023-01-22
6,1,450,3,150.000000,2023-01-29
7,2,250,3,83.333333,2023-01-29
8,3,1150,5,230.000000,2023-01-29


### Exercise 4
Create:
- `days_since_last_event`

Rules:
- Use snapshot_time
- Users with no prior events -> NaN

Explain why recency is powerful.

In [7]:
recency_rows = []

for snapshot_time in SNAPSHOT_DATES:
    past_events = events[events["event_time"] < snapshot_time]

    last_event = (
        past_events
        .groupby("user_id")["event_time"]
        .max()
        .reset_index(name="last_event_time")
    )

    last_event["days_since_last_event"] = (
        snapshot_time - last_event["last_event_time"]
    ).dt.days

    last_event["snapshot_time"] = snapshot_time
    recency_rows.append(last_event)

recency_features = pd.concat(recency_rows, ignore_index=True)
recency_features

,user_id,last_event_time,days_since_last_event,snapshot_time
0,1,2023-01-10,5,2023-01-15
1,2,2023-01-07,8,2023-01-15
2,3,2023-01-14,1,2023-01-15
3,1,2023-01-20,2,2023-01-22
4,2,2023-01-15,7,2023-01-22
5,3,2023-01-18,4,2023-01-22
6,1,2023-01-20,9,2023-01-29
7,2,2023-01-25,4,2023-01-29
8,3,2023-01-28,1,2023-01-29


Why recency is powerful
- Captures **engagement freshness**
- Often stronger than raw counts
- Used heavily in churn & fraud models

### Exercise 5
On the **events table:**
- Create `prev_amount` per user
- Validate NaNs appear

Explain:
- What it means if NaNs don’t appear

In [8]:
events_sorted = events.sort_values(["user_id", "event_time"])

events_sorted["prev_amount"] = (
    events_sorted
    .groupby("user_id")["amount"]
    .shift(1)
)
events_sorted

,event_id,user_id,event_time,amount,prev_amount
0,101,1,2023-01-03,100,NaN
1,102,1,2023-01-10,200,100.0
2,103,1,2023-01-20,150,200.0
3,104,2,2023-01-07,50,NaN
4,105,2,2023-01-15,80,50.0
5,106,2,2023-01-25,120,80.0
6,107,3,2023-01-12,300,NaN
7,108,3,2023-01-14,400,300.0
8,109,3,2023-01-18,200,400.0
9,110,3,2023-01-22,100,200.0


### Exercise 6
For each snapshot:
- Compute rolling_7d_amount

Rules:
- Use **time-based rolling**
- Respect snapshot cutoff
- Reduce to **one value per user per snapshot**

In [13]:
events_sorted = (
    events
    .sort_values(["user_id", "event_time"])
    .set_index("event_time")
)
events_sorted

,event_id,user_id,amount
event_time,,,
2023-01-03,101,1,100
2023-01-10,102,1,200
2023-01-20,103,1,150
2023-01-07,104,2,50
2023-01-15,105,2,80
2023-01-25,106,2,120
2023-01-12,107,3,300
2023-01-14,108,3,400
2023-01-18,109,3,200


In [14]:
rolling_series = (
    events_sorted
    .groupby("user_id")["amount"]
    .rolling("7D")
    .sum()
    .reset_index(level=0, drop=True)  # 🔑 CRITICAL FIX
)

events_sorted["rolling_7d_amount"] = rolling_series
events_sorted

,event_id,user_id,amount,rolling_7d_amount
event_time,,,,
2023-01-03,101,1,100,100.0
2023-01-10,102,1,200,200.0
2023-01-20,103,1,150,150.0
2023-01-07,104,2,50,50.0
2023-01-15,105,2,80,80.0
2023-01-25,106,2,120,120.0
2023-01-12,107,3,300,300.0
2023-01-14,108,3,400,700.0
2023-01-18,109,3,200,900.0


### Exercise 7
Create label:
```text
active_next_7d
```

Definition:
- 1 if user has **any event**
```text
snapshot_time <= event_time < snapshot_time + 7 days
```
- 0 otherwise

In [15]:
label_rows = []

for snapshot_time in SNAPSHOT_DATES:
    future_events = events[
        (events["event_time"] >= snapshot_time) &
        (events["event_time"] < snapshot_time + pd.Timedelta(days=7))
    ]

    labels = (
        future_events
        .groupby("user_id")
        .size()
        .gt(0)
        .astype(int)
        .reset_index(name="active_next_7d")
    )

    labels["snapshot_time"] = snapshot_time
    label_rows.append(labels)

labels_df = pd.concat(label_rows, ignore_index=True)
labels_df

,user_id,active_next_7d,snapshot_time
0,1,1,2023-01-15
1,2,1,2023-01-15
2,3,1,2023-01-15
3,2,1,2023-01-22
4,3,1,2023-01-22
